# Google Colab — Octopus EXP045-equivalent WIDTH512 @50k

Это CUDA-версия нашего фиксированного Octopus benchmark.

**Научный рецепт не меняется** относительно moth:
WIDTH512 / RF1029, batch=4, seed=42, BCE + 0.1×Huber, 25k @1e-3,
затем cosine 3e-4→3e-5 до 50k.

Меняется только backend: Windows DirectML → Google Colab CUDA.
Поэтому результат не обязан быть бит-в-бит идентичен локальному запуску,
но sampler, objective, optimizer implementation и schedule сохраняются.

Перед запуском: `Runtime → Change runtime type → GPU`.


In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Upload octopus_colab_bundle.zip into this Drive folder.
DRIVE_ROOT = Path("/content/drive/MyDrive/EPIC_COLAB")
BUNDLE_PATH = DRIVE_ROOT / "octopus_colab_bundle.zip"

# Persistent outputs/checkpoints live on Drive.
DRIVE_ARTIFACT_ROOT = DRIVE_ROOT / "octopus_cuda_artifacts"
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "octopus_processed_split"

assert BUNDLE_PATH.is_file(), (
    f"Upload bundle first: {BUNDLE_PATH}"
)

DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print("Bundle:", BUNDLE_PATH)
print("Artifacts:", DRIVE_ARTIFACT_ROOT)


In [ ]:
# Extract project/data to fast local Colab storage.
import shutil, zipfile, os

CONTENT_ROOT = Path("/content")
PROJECT_ROOT = CONTENT_ROOT / "EPIC"
EXTRACT_MARKER = PROJECT_ROOT / ".octopus_bundle_extracted"

if not EXTRACT_MARKER.is_file():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    with zipfile.ZipFile(BUNDLE_PATH, "r") as zf:
        zf.extractall(CONTENT_ROOT)
    EXTRACT_MARKER.write_text("ok\n", encoding="utf-8")

assert (PROJECT_ROOT / "README.md").is_file()
assert (PROJECT_ROOT / "src/ml_project").is_dir()
assert (PROJECT_ROOT / "data/raw/unpacked/ASM119413v2").is_dir()

# Persist prepared split on Drive, while raw FASTA stays on fast /content.
local_processed = PROJECT_ROOT / "data/processed"
if local_processed.exists() and not local_processed.is_symlink():
    shutil.rmtree(local_processed)
if not local_processed.exists():
    os.symlink(DRIVE_PROCESSED_ROOT, local_processed, target_is_directory=True)

print("Project:", PROJECT_ROOT)
print("Processed ->", local_processed.resolve())


In [ ]:
# Environment / integrity checks.
import copy, gc, hashlib, json, math, sys, time, warnings
import numpy as np
import pandas as pd
import torch

EXPECTED_AGENTS_SHA = "bbc36fcf2270d4ce806e25a5be0d6b7eeab5fe80bc91acd040e71459baac68c3"
EXPECTED_LOADER_SHA = "37c9b0ed4ccecc02b484010c7b5b02fa35fa35f4c976e76e6d0e00096380b468"

def sha256_file(path, block=8 * 1024 * 1024):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            d.update(chunk)
    return d.hexdigest()

assert torch.cuda.is_available(), "GPU runtime is required"
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True

AGENTS = PROJECT_ROOT / "AGENTS.md"
LOADER = PROJECT_ROOT / "src/ml_project/epic_data.py"

assert sha256_file(AGENTS) == EXPECTED_AGENTS_SHA
assert sha256_file(LOADER) == EXPECTED_LOADER_SHA

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,  # implementation is device-agnostic; we keep exact Adam math
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    evaluate_presence_intensity_model,
    forward_both_strands_multitask,
)
from ml_project.epic_data import SplitConfig, ensure_prepared_split, load_split

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("VRAM GiB:", round(props.total_memory / 1024**3, 2))


In [ ]:
# Prepare/load deterministic Octopus split.
ASSEMBLY = "ASM119413v2"
SPLIT_CONFIG = SplitConfig(
    assembly=ASSEMBLY,
    split_version="contig_holdout_v1",
    seed=42,
    validation_contig_fraction=0.25,
)

split = ensure_prepared_split(PROJECT_ROOT, SPLIT_CONFIG)
split = load_split(PROJECT_ROOT, SPLIT_CONFIG, verify_hashes=True)

display(split.summary)
display(split.contig_assignment)

train_row = split.summary.loc[split.summary["split"]=="train"].iloc[0]
val_row = split.summary.loc[split.summary["split"]=="validation"].iloc[0]
test_row = split.summary.loc[split.summary["split"]=="test"].iloc[0]

assert int(train_row["contigs"]) > 0
assert int(val_row["contigs"]) > 0
assert int(test_row["contigs"]) > 0
assert int(train_row["positive_positions"]) > 0
assert int(val_row["positive_positions"]) > 0
assert split.positive_targets("test") is None

EXPECTED_VAL_POSITIONS = int(val_row["position_strand_count"])
EXPECTED_VAL_POSITIVES = int(val_row["positive_positions"])
EXPECTED_TEST_POSITIONS = int(test_row["position_strand_count"])

assert EXPECTED_TEST_POSITIONS == 416_885_800, (
    EXPECTED_TEST_POSITIONS,
    416_885_800,
)

print("Split: PASS")
print("Train positives:", f"{int(train_row['positive_positions']):,}")
print("Val positives:", f"{EXPECTED_VAL_POSITIVES:,}")
print("Val position-strand:", f"{EXPECTED_VAL_POSITIONS:,}")
print("Official test position-strand:", f"{EXPECTED_TEST_POSITIONS:,}")


In [ ]:
# Load compact uint8 FASTA arrays into RAM and smoke-test geometry.
sequences, fasta_alphabet = load_baseline_sequences(split)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

probe_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
probe = probe_generator.sample_batch(4)

assert tuple(probe.x.shape) == (4,5,1284)
assert tuple(probe.x_both_strands.shape) == (8,5,1284)
assert tuple(probe.target.shape) == (4,2,1024)
assert tuple(probe.count.shape) == (4,2,1024)
assert tuple(probe.mask.shape) == (4,2,1024)

model_probe = create_cnn_presence_intensity(channels=512)
n_params = sum(p.numel() for p in model_probe.parameters())
assert n_params == 14_708_738

print("Geometry:", tuple(probe.x_both_strands.shape))
print("Parameters:", f"{n_params:,}")
print("OCTOPUS CUDA READINESS: PASS")

del probe, probe_generator, model_probe
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Frozen EXP045-equivalent recipe.
EXPERIMENT_ID = "SUBMIT-OCTOPUS-EXP045-CUDA"
EXPERIMENT_SLUG = "SUBMIT_ASM119413v2_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029_CUDA"

SEED = 42
STEPS = 50_000
PHASE1_END = 25_000
BATCH_SIZE = 4
LOG_EVERY = 250
CHECKPOINT_STEPS = {10_000,25_000,30_000,35_000,40_000,45_000,50_000}

PHASE1_LR = 1e-3
PHASE2_START_LR = 3e-4
PHASE2_END_LR = 3e-5
PHASE2_STEPS = STEPS - PHASE1_END
INTENSITY_WEIGHT = 0.1

RUN_DIR = DRIVE_ARTIFACT_ROOT / EXPERIMENT_SLUG / "run_001"
RUN_DIR.mkdir(parents=True, exist_ok=True)

FINAL = RUN_DIR / "width512_50k_final.pt"
BRANCH = RUN_DIR / "model_step_50000.pt"
HISTORY = RUN_DIR / "training_history.csv"
METRICS = RUN_DIR / "validation_metrics_report.csv"
SUMMARY = RUN_DIR / "result_summary.json"

def model_state_on_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def tree_to_cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k:tree_to_cpu(v) for k,v in x.items()}
    if isinstance(x, list):
        return [tree_to_cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple(tree_to_cpu(v) for v in x)
    return x

def ckpt_path(step):
    return RUN_DIR / f"model_step_{step:05d}.pt"

def atomic_torch_save(path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def save_ckpt(path, step, model, opt, sched, gen, history, seen, elapsed):
    payload = {
        "experiment": EXPERIMENT_ID,
        "assembly": ASSEMBLY,
        "backend": "cuda",
        "gpu_name": torch.cuda.get_device_name(0),
        "model_type": "presence_intensity",
        "architecture": "rf1029_width512",
        "step": int(step),
        "model_state_dict": model_state_on_cpu(model),
        "optimizer_state_dict": tree_to_cpu(opt.state_dict()),
        "scheduler_state_dict": None if sched is None else tree_to_cpu(sched.state_dict()),
        "generator_rng_state": copy.deepcopy(gen.rng.bit_generator.state),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": [x.cpu() for x in torch.cuda.get_rng_state_all()],
        "model_config": {
            "input_channels":5,
            "channels":512,
            "output_channels":2,
            "theoretical_rf":1029,
        },
        "profile_config": {
            "target_length":1024,
            "context_flank":130,
            "positive_branch_fraction":0.5,
        },
        "training_config": {
            "seed":SEED,
            "steps":STEPS,
            "batch_size":BATCH_SIZE,
            "phase1_end_step":PHASE1_END,
            "phase1_learning_rate":PHASE1_LR,
            "phase2_start_lr":PHASE2_START_LR,
            "phase2_end_lr":PHASE2_END_LR,
            "phase2_scheduler":"CosineAnnealingLR",
            "intensity_loss_weight":INTENSITY_WEIGHT,
            "huber_delta":1.0,
            "optimizer":"DirectMLAdam implementation on CUDA",
        },
        "loader_sha256": sha256_file(LOADER),
        "seen_position_strand": int(seen),
        "elapsed_seconds": float(elapsed),
        "history": list(history),
    }
    atomic_torch_save(path, payload)

print("Persistent run dir:", RUN_DIR)


## Long training — 50k

Checkpoints пишутся прямо в Google Drive. Если Colab оборвётся,
повторный запуск notebook продолжит с последнего checkpoint.


In [ ]:
if FINAL.is_file():
    done = torch.load(FINAL, map_location="cpu", weights_only=False)
    assert int(done["step"]) == 50_000
    assert done["assembly"] == ASSEMBLY
    print("Already complete:", FINAL)
    del done

else:
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    generator = ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=SEED,
    )

    model = create_cnn_presence_intensity(channels=512).to(device)
    optimizer = DirectMLAdam(model.parameters(), lr=PHASE1_LR)
    scheduler = None

    available = [
        (s, ckpt_path(s))
        for s in sorted(CHECKPOINT_STEPS)
        if ckpt_path(s).is_file()
    ]
    start_step, resume_path = available[-1] if available else (0, None)

    history = []
    seen = 0
    previous_elapsed = 0.0

    if resume_path is not None:
        resume = torch.load(resume_path, map_location="cpu", weights_only=False)
        assert int(resume["step"]) == start_step
        assert resume["assembly"] == ASSEMBLY

        model.load_state_dict(resume["model_state_dict"])
        optimizer.load_state_dict(resume["optimizer_state_dict"])
        generator.rng.bit_generator.state = copy.deepcopy(
            resume["generator_rng_state"]
        )
        torch.set_rng_state(resume["torch_rng_state"])
        if "cuda_rng_state_all" in resume:
            torch.cuda.set_rng_state_all(
                [x.cpu() for x in resume["cuda_rng_state_all"]]
            )

        history = list(resume.get("history", []))
        seen = int(resume.get("seen_position_strand", 0))
        previous_elapsed = float(resume.get("elapsed_seconds", 0.0))

        if start_step < PHASE1_END:
            for g in optimizer.param_groups:
                g["lr"] = PHASE1_LR
        elif start_step == PHASE1_END:
            for g in optimizer.param_groups:
                g["lr"] = PHASE2_START_LR
                g["initial_lr"] = PHASE2_START_LR
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=PHASE2_STEPS,
                eta_min=PHASE2_END_LR,
            )
        else:
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=PHASE2_STEPS,
                eta_min=PHASE2_END_LR,
            )
            scheduler.load_state_dict(resume["scheduler_state_dict"])

        del resume
        gc.collect()
        torch.cuda.empty_cache()
        print("RESUME:", resume_path)

    ema_p = history[-1]["ema_presence"] if history else None
    ema_i = history[-1]["ema_intensity"] if history else None
    ema_t = history[-1]["ema_total"] if history else None

    started = time.perf_counter()
    model.train()

    for step in range(start_step + 1, STEPS + 1):
        batch = generator.sample_batch(BATCH_SIZE)

        x = batch.x_both_strands.to(device, non_blocking=True)
        pt = batch.plus_target.to(device, non_blocking=True)
        mt = batch.minus_target_reverse.to(device, non_blocking=True)
        pw = batch.plus_loss_weight.to(device, non_blocking=True)
        mw = batch.minus_loss_weight_reverse.to(device, non_blocking=True)

        pc = batch.count[:,0:1,:].contiguous()
        mc = batch.count[:,1:2,:].flip(-1).contiguous()

        pit = torch.log1p(pc).to(device, non_blocking=True)
        mit = torch.log1p(mc).to(device, non_blocking=True)
        piw = (
            2.0 * batch.plus_loss_weight * (pc > 0).float()
        ).to(device, non_blocking=True)
        miw = (
            2.0 * batch.minus_loss_weight_reverse * (mc > 0).float()
        ).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        pp, mp, pi, mi = forward_both_strands_multitask(
            model,
            x,
            batch_size=BATCH_SIZE,
            context_flank=130,
            target_length=1024,
        )

        ploss = dml_weighted_profile_bce_with_logits(
            pp, mp, pt, mt, pw, mw
        )
        iloss = (
            dml_weighted_huber(pi, pit, piw, delta=1.0)
            + dml_weighted_huber(mi, mit, miw, delta=1.0)
        )
        loss = ploss + INTENSITY_WEIGHT * iloss

        pv = float(ploss.detach().cpu().item())
        iv = float(iloss.detach().cpu().item())
        tv = float(loss.detach().cpu().item())
        assert all(math.isfinite(v) for v in (pv, iv, tv))

        loss.backward()
        optimizer.step()

        if step == PHASE1_END:
            for g in optimizer.param_groups:
                g["lr"] = PHASE2_START_LR
                g["initial_lr"] = PHASE2_START_LR
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=PHASE2_STEPS,
                eta_min=PHASE2_END_LR,
            )
        elif step > PHASE1_END:
            scheduler.step()

        ema_p = pv if ema_p is None else 0.99 * ema_p + 0.01 * pv
        ema_i = iv if ema_i is None else 0.99 * ema_i + 0.01 * iv
        ema_t = tv if ema_t is None else 0.99 * ema_t + 0.01 * tv
        seen += int(batch.mask.sum())

        elapsed_process = time.perf_counter() - started
        elapsed_total = previous_elapsed + elapsed_process
        sec_step = elapsed_process / max(step - start_step, 1)

        if step % LOG_EVERY == 0 or step == STEPS:
            rec = {
                "step":step,
                "learning_rate":float(optimizer.param_groups[0]["lr"]),
                "presence_loss":pv,
                "intensity_loss":iv,
                "total_loss":tv,
                "ema_presence":float(ema_p),
                "ema_intensity":float(ema_i),
                "ema_total":float(ema_t),
                "elapsed_seconds":float(elapsed_total),
                "seconds_per_step_current_process":float(sec_step),
                "estimated_training_hours_remaining":float(
                    (STEPS-step) * sec_step / 3600
                ),
                "seen_position_strand":int(seen),
            }
            history.append(rec)
            pd.DataFrame(history).to_csv(HISTORY, index=False)
            print(
                f"step={step:05d}/{STEPS} "
                f"lr={optimizer.param_groups[0]['lr']:.8f} "
                f"loss=({pv:.4f},{iv:.4f},{tv:.4f}) "
                f"sec/step={sec_step:.3f} "
                f"ETA={(STEPS-step)*sec_step/3600:.2f}h",
                flush=True,
            )

        if step in CHECKPOINT_STEPS:
            save_ckpt(
                ckpt_path(step),
                step,
                model,
                optimizer,
                scheduler,
                generator,
                history,
                seen,
                elapsed_total,
            )

        del (
            batch,x,pt,mt,pw,mw,pc,mc,pit,mit,piw,miw,
            pp,mp,pi,mi,ploss,iloss,loss
        )

    elapsed_total = previous_elapsed + (time.perf_counter() - started)
    assert scheduler is not None
    assert math.isclose(
        float(optimizer.param_groups[0]["lr"]),
        PHASE2_END_LR,
        rel_tol=1e-6,
        abs_tol=1e-10,
    )

    save_ckpt(
        FINAL, STEPS, model, optimizer, scheduler,
        generator, history, seen, elapsed_total
    )

    payload = torch.load(FINAL, map_location="cpu", weights_only=False)
    atomic_torch_save(BRANCH, payload)
    del payload

    print("TRAINING COMPLETE")
    print("Hours:", round(elapsed_total / 3600, 3))
    print("Branch point:", BRANCH)

    del model, optimizer, scheduler, generator
    gc.collect()
    torch.cuda.empty_cache()


## Full natural validation @50k

Validation используется только для оценки после завершения 50k.


In [ ]:
checkpoint = torch.load(BRANCH, map_location="cpu", weights_only=False)
assert checkpoint["assembly"] == ASSEMBLY
assert int(checkpoint["step"]) == 50_000

model = create_cnn_presence_intensity(channels=512)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

val_gen = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

result = evaluate_presence_intensity_model(
    model,
    val_gen,
    device=device,
    batch_size=1,
    score_decimals=5,
    progress_every_batches=100,
    progress=print,
    # DirectML-specific warning detector is irrelevant on CUDA.
    fail_on_cpu_fallback=False,
)

display(result.metrics)
result.metrics.to_csv(METRICS, index=False)

overall = result.metrics.loc[
    (result.metrics["score"]=="presence_probability")
    & (result.metrics["segment"]=="overall")
].iloc[0]

summary = {
    "experiment":EXPERIMENT_ID,
    "assembly":ASSEMBLY,
    "backend":"cuda",
    "gpu_name":torch.cuda.get_device_name(0),
    "checkpoint":str(BRANCH),
    "checkpoint_sha256":sha256_file(BRANCH),
    "step":50_000,
    "validation_position_strand":int(result.processed_positions),
    "validation_positives":int(result.processed_positives),
    "presence_average_precision_5dp":float(overall["average_precision"]),
    "presence_epic_spearman":float(overall["epic_spearman"]),
    "validation_elapsed_seconds":float(result.elapsed_seconds),
    "validation_positions_per_second":float(result.positions_per_second),
    "next_stage":"octopus EXP047-B train-only risk cache +20k",
}

SUMMARY.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

assert result.processed_positions == EXPECTED_VAL_POSITIONS
assert result.processed_positives == EXPECTED_VAL_POSITIVES

print("OCTOPUS EXP045 CUDA: COMPLETE")
print("Send me:", SUMMARY)

del checkpoint, model, val_gen, result
gc.collect()
torch.cuda.empty_cache()
